# Notebook D — Aggregate Results & Reproduce Tables 2-6 + Figures 3-6

This notebook does no training or cryptography — it only reads the result
pickles produced by Notebooks A, B, C and by the original `BIAB.ipynb` /
`BIAB2.ipynb` / `BIAB3.ipynb` (fixed version) runs, and prints Tables 2-6 and
saves Figures 3-6.

**Assumed available result files** in `/content/drive/MyDrive/path1_results/`:

From Notebook A: `B1_centralised_if`, `B3_fl_local_anomaly`, `B4_flit`
From Notebook B: `Table3_B2_FedAvg_XXpct` (5), `Table3_BiAB_IoT_XXpct` (5),
                 `Table4_ablation_{P1_only,P2_only,P1_plus_P2}`
From Notebook C: `Table5_ecdsa_microbench`, `Table5_pbft_kX_fY_pZZ`

**Plus the three baseline configurations from the original notebooks** —
save these once by hand at the end of BIAB / BIAB2 / BIAB3 with e.g.:

```python
from biab_common import save_result, evaluate
metrics = evaluate(global_model, X_test, y_test)
save_result('Table2_baseline_clean', metrics, {'method':'FedAvg-clean'})
```

Then re-run this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install matplotlib pandas numpy
import sys, os
sys.path.insert(0, '/content/drive/MyDrive/path1_code')
from biab_common import load_all_results, RESULT_DIR
import numpy as np, pandas as pd, matplotlib.pyplot as plt

def get(name):
    for r in load_all_results():
        if r['name'] == name:
            return r
    print(f'  WARNING: {name} not found in {RESULT_DIR}')
    return None

def pct(x, digits=2):
    return f'{x*100:.{digits}f}' if x is not None else '---'

## Table 2 — Anomaly Detection Performance Comparison

Rows: B1 (Centralised IF), B2 (FedAvg baseline @0%), B3 (FL + local anomaly),
B4 (FLIT), BiAB-IoT (P1+P2).

In [ ]:
rows = []
for label, name in [
    ('Centralised Isolation Forest (B1)',  'B1_centralised_if'),
    ('Standard FedAvg (B2)',                'Table3_B2_FedAvg_00pct'),
    ('FL + local anomaly (B3)',             'B3_fl_local_anomaly'),
    ('FLIT (B4)',                            'B4_flit'),
    ('Proposed BiAB-IoT (P1+P2) @0%',       'Table3_BiAB_IoT_00pct'),
]:
    r = get(name)
    if r is None: continue
    m = r['metrics']
    rows.append({
        'Model': label,
        'Accuracy (%)':  pct(m['accuracy']),
        'Precision (%)': pct(m['precision']),
        'Recall (%)':    pct(m['recall']),
        'F1 (%)':        pct(m['f1']),
        'FPR (%)':       pct(m['fpr']),
    })
df_t2 = pd.DataFrame(rows)
print('\nTable 2 — Anomaly Detection Performance Comparison')
print(df_t2.to_string(index=False))
df_t2.to_csv(os.path.join(RESULT_DIR, 'Table_2.csv'), index=False)

## Table 3 — Accuracy under label-flipping poisoning

In [ ]:
ATTACK_PCTS = [0, 5, 10, 15, 20]
rows = []
for method_label, name_tpl in [
    ('B2: FedAvg',        'Table3_B2_FedAvg_{:02d}pct'),
    ('BiAB-IoT (P1+P2)',  'Table3_BiAB_IoT_{:02d}pct'),
]:
    row = {'Method': method_label}
    for p in ATTACK_PCTS:
        r = get(name_tpl.format(p))
        row[f'{p}%'] = pct(r['metrics']['accuracy']) if r else '---'
    rows.append(row)

df_t3 = pd.DataFrame(rows)
print('\nTable 3 — Accuracy (%) Under Label-Flipping Poisoning')
print(df_t3.to_string(index=False))
df_t3.to_csv(os.path.join(RESULT_DIR, 'Table_3.csv'), index=False)

In [ ]:
# --- Figure 3: Table 3 as a line plot ------------------------------------
plt.figure(figsize=(8, 5))
for method_label, name_tpl, marker in [
    ('B2: FedAvg', 'Table3_B2_FedAvg_{:02d}pct', 's'),
    ('BiAB-IoT (Proposed)', 'Table3_BiAB_IoT_{:02d}pct', '*'),
]:
    ys = []
    for p in ATTACK_PCTS:
        r = get(name_tpl.format(p))
        ys.append(r['metrics']['accuracy'] * 100 if r else np.nan)
    plt.plot(ATTACK_PCTS, ys, marker=marker, linewidth=2, label=method_label)
plt.xlabel('Malicious Client Percentage (%)')
plt.ylabel('Detection Accuracy (%)')
plt.title('Figure 3: Accuracy under Label-Flipping Poisoning Attacks (Table 3)')
plt.legend(); plt.grid(alpha=0.4); plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'Figure_3.png'), dpi=200)
plt.show()

## Table 4 — Ablation study (10% malicious, single condition)

In [ ]:
rows = []
for label, name in [
    ('Base FedAvg (no P1, no P2)', 'Table3_B2_FedAvg_10pct'),
    ('P1 only (AI->Blockchain)',   'Table4_ablation_P1_only'),
    ('P2 only (Blockchain->AI)',   'Table4_ablation_P2_only'),
    ('P1 + P2 (Full BiAB-IoT)',    'Table4_ablation_P1_plus_P2'),
]:
    r = get(name)
    if r is None: continue
    m = r['metrics']
    rows.append({
        'Configuration': label,
        'Accuracy (%)': pct(m['accuracy']),
        'F1 (%)':       pct(m['f1']),
        'FPR (%)':      pct(m['fpr']),
    })

df_t4 = pd.DataFrame(rows)
# Add "gain over baseline" column
if len(rows) >= 4:
    baseline = float(rows[0]['Accuracy (%)'])
    df_t4['Gain over Baseline (pp)'] = [
        f'{float(r["Accuracy (%)"]) - baseline:+.2f}' for r in rows]
print('\nTable 4 — Ablation Study Results')
print(df_t4.to_string(index=False))
df_t4.to_csv(os.path.join(RESULT_DIR, 'Table_4.csv'), index=False)

# --- Figure 4: bar chart of ablation configurations ---------------------
if len(rows) == 4:
    plt.figure(figsize=(8, 5))
    x = np.arange(len(rows))
    width = 0.35
    accs = [float(r['Accuracy (%)']) for r in rows]
    f1s  = [float(r['F1 (%)'])       for r in rows]
    plt.bar(x - width/2, accs, width, label='Accuracy (%)')
    plt.bar(x + width/2, f1s,  width, label='F1-Score (%)')
    plt.xticks(x, [r['Configuration'] for r in rows], rotation=20, ha='right')
    plt.ylabel('Score (%)')
    plt.title('Figure 4: Ablation Study — Contribution of P1 and P2 Mechanisms')
    plt.legend(); plt.grid(axis='y', alpha=0.4); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'Figure_4.png'), dpi=200)
    plt.show()

## Table 5 — PBFT Consensus Latency with ECDSA P-256

In [ ]:
ecdsa = get('Table5_ecdsa_microbench')
if ecdsa:
    e = ecdsa['metrics']
    print(f'ECDSA sign:   mean {e["sign_mean_ms"]:.2f} ms, std {e["sign_std_ms"]:.2f} ms')
    print(f'ECDSA verify: mean {e["verify_mean_ms"]:.2f} ms, std {e["verify_std_ms"]:.2f} ms')

rows = []
for k, f, p in [(4, 1, 5), (4, 1, 10), (4, 1, 20), (7, 2, 5), (7, 2, 10)]:
    r = get(f'Table5_pbft_k{k}_f{f}_p{p:02d}')
    if r is None: continue
    m = r['metrics']
    rows.append({
        f'Validators (f)': f'{k} (f={f})',
        'Proposals / Window': p,
        'Mean Latency (ms)': f'{m["mean_ms"]:.0f}',
        '95th Percentile (ms)': f'{m["p95_ms"]:.0f}',
    })
df_t5 = pd.DataFrame(rows)
print('\nTable 5 — PBFT Consensus Latency with ECDSA P-256 Overhead')
print(df_t5.to_string(index=False))
df_t5.to_csv(os.path.join(RESULT_DIR, 'Table_5.csv'), index=False)

# --- Figure 5: latency bar chart ---------------------------------------
if rows:
    plt.figure(figsize=(9, 5))
    x = np.arange(len(rows))
    means = [float(r['Mean Latency (ms)']) for r in rows]
    p95s  = [float(r['95th Percentile (ms)']) for r in rows]
    plt.bar(x - 0.2, means, 0.4, label='Mean')
    plt.bar(x + 0.2, p95s,  0.4, label='95th percentile')
    labels = [f'{r[list(r.keys())[0]]}\n{r["Proposals / Window"]} prop/win' for r in rows]
    plt.xticks(x, labels)
    plt.ylabel('Latency (ms)')
    plt.title('Figure 5: PBFT Consensus Latency with ECDSA P-256 Overhead (Table 5)')
    plt.axhline(500, color='red', linestyle='--', linewidth=1,
                label='500 ms threshold (acceptable for industrial IoT)')
    plt.legend(); plt.grid(axis='y', alpha=0.4); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'Figure_5.png'), dpi=200)
    plt.show()

## Table 6 — Comprehensive comparison across frameworks & Figure 6 (radar)

In [ ]:
def poisoning_resilience(name_tpl):
    """Mean accuracy across the 5 attack percentages -> single resilience metric."""
    ys = []
    for p in ATTACK_PCTS:
        r = get(name_tpl.format(p))
        if r: ys.append(r['metrics']['accuracy'])
    return np.mean(ys) if ys else None

rows = []
# Reuse zero-attack row for the 'clean accuracy' column
for framework, clean_name, sweep_tpl in [
    ('Standard FedAvg', 'Table3_B2_FedAvg_00pct',   'Table3_B2_FedAvg_{:02d}pct'),
    ('BiAB-IoT (Proposed)', 'Table3_BiAB_IoT_00pct', 'Table3_BiAB_IoT_{:02d}pct'),
]:
    r = get(clean_name)
    if r is None: continue
    m = r['metrics']
    res = poisoning_resilience(sweep_tpl)
    rows.append({
        'Framework': framework,
        'Detection Accuracy (%)': pct(m['accuracy']),
        'Poisoning Resilience (%)': pct(res) if res else '---',
        'Automated Enforcement': '✓' if 'BiAB' in framework else '✗',
        'Bidirectional Feedback': 'Bidirectional (AI<->B)' if 'BiAB' in framework else '✗',
        'Consensus Mechanism': 'PBFT' if 'BiAB' in framework else 'None',
    })

# Also include B1, B3, B4 at clean condition for a full-column view
for label, name, extra in [
    ('BC-FL [18]', 'B4_flit', {'Automated Enforcement':'✗',
                                'Bidirectional Feedback':'✗',
                                'Consensus Mechanism':'PBFT'}),
]:
    r = get(name)
    if r is None: continue
    m = r['metrics']
    row = {'Framework': label,
           'Detection Accuracy (%)': pct(m['accuracy']),
           'Poisoning Resilience (%)': pct(m['accuracy']),
           **extra}
    rows.insert(-1, row)   # keep BiAB-IoT last

df_t6 = pd.DataFrame(rows)
print('\nTable 6 — Comprehensive Comparison')
print(df_t6.to_string(index=False))
df_t6.to_csv(os.path.join(RESULT_DIR, 'Table_6.csv'), index=False)

In [ ]:
# --- Figure 6: radar chart across 5 metrics ---------------------------
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', '1-FPR']
methods = [
    ('Centralised IF', 'B1_centralised_if'),
    ('FedAvg',         'Table3_B2_FedAvg_00pct'),
    ('FLIT',           'B4_flit'),
    ('BiAB-IoT',       'Table3_BiAB_IoT_00pct'),
]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, polar=True)
angles = np.linspace(0, 2*np.pi, len(metrics_names), endpoint=False).tolist()
angles += angles[:1]

for label, name in methods:
    r = get(name)
    if r is None: continue
    m = r['metrics']
    vals = [m['accuracy']*100, m['precision']*100,
            m['recall']*100,   m['f1']*100,
            (1 - m['fpr'])*100]
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=2, label=label)
    ax.fill(angles, vals, alpha=0.10)

ax.set_theta_offset(np.pi / 2); ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics_names)
ax.set_ylim(70, 100)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.title('Figure 6: Radar Chart — Multi-Metric Comparison across Methods')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'Figure_6.png'), dpi=200)
plt.show()

print(f'\nAll tables and figures written to {RESULT_DIR}')

## Next steps

* Copy the five CSVs (`Table_2.csv` ... `Table_6.csv`) and the four PNGs
  (`Figure_3.png`, `Figure_4.png`, `Figure_5.png`, `Figure_6.png`) off Drive.
* Send me the numbers and I will:
  1. Replace the corresponding rows/cells in `Manuscript_Revised.docx`.
  2. Rewrite the abstract, discussion, and future work to reflect what the
     measurements show (which may or may not be the story the original
     draft told — we'll see when the numbers arrive).
  3. Produce a new tracked-changes doc so the delta from the current v2
     submission is visible.
